# Optional ETL / Validation Notebook

This notebook runs final validation checks on the reporting layer before dashboard publishing.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

repo_root = Path.cwd().parents[0] if Path.cwd().name == 'python' else Path.cwd()
db_path = repo_root / 'data' / 'processed' / 'olist_reporting.duckdb'
con = duckdb.connect(str(db_path))
print('Connected:', db_path)

In [ ]:
row_counts = con.execute("""
SELECT *
FROM read_csv_auto('data/processed/model_row_counts.csv', header=true)
ORDER BY table_name
""").df()
row_counts

In [ ]:
headline = con.execute("""
SELECT
    COUNT(*) AS total_orders,
    SUM(COALESCE(item_gmv, 0)) AS total_gmv,
    AVG(COALESCE(item_gmv, 0)) AS aov,
    AVG(CASE WHEN is_canceled_or_unavailable = 1 THEN 1.0 ELSE 0.0 END) AS cancellation_rate,
    AVG(avg_review_score) AS avg_review_score,
    AVG(delivery_days) AS avg_delivery_days,
    AVG(CASE WHEN is_on_time_delivery IS NULL THEN NULL ELSE is_on_time_delivery::DOUBLE END) AS on_time_delivery_rate
FROM marts.fact_orders
""").df()
headline

In [ ]:
validation = con.execute("""
WITH by_items AS (
    SELECT SUM(gmv) AS item_level_gmv
    FROM marts.fact_order_items
),
by_orders AS (
    SELECT SUM(item_gmv) AS order_level_gmv
    FROM marts.fact_orders
)
SELECT
    item_level_gmv,
    order_level_gmv,
    item_level_gmv - order_level_gmv AS difference_should_be_zero
FROM by_items CROSS JOIN by_orders
""").df()
validation